# 14 · Phases 12 & 15 — Agent selection and RL-level ablations

1. **Select the agent** that drives the final model: the tabular method with the best mean
   validation reward over seeds (Notebooks 11–12); within it, the seed with the best validation
   reward. The test split is *not* used for any choice.
2. **Ablations at the policy level** (proxy classifier, 3 seeds each, same agent and budget):
   remove RL entirely; fix the wavelet / band / window to the baseline value; drop the wavelet
   branch; accuracy-only reward; no robustness term; no complexity term; no state adaptation (K = 1);
   rewards from the classical oracle (Notebook 06) instead of the deep supernet oracle (Notebook 07).
   Every variant's policy is *evaluated* with the deep oracle, so rows are comparable.
3. Export each variant's policy for the deep-model experiments (Notebooks 16–17).

In [ ]:
%run common/00_config.ipynb
%run common/01_signal_lib.ipynb
%run common/02_rl_lib.ipynb
import joblib
import matplotlib.pyplot as plt

K = load_json(P["results"] / "chosen_K.json")["K"]
prob = RLProblem(CFG, P, K=K)
R = CFG["rl"]
runs = joblib.load(P["processed"] / "tabular_runs.joblib")
rows = [{"method": m, "seed": s, "val_reward": prob.evaluate(v["policy"], "val")["mean_reward"]} for (m, s), v in runs.items()]
sel = pd.DataFrame(rows)
by_method = sel.groupby("method").val_reward.agg(["mean", "std"]).sort_values("mean", ascending=False)
by_method

In [ ]:
LEARNERS = {"mc": ("mc", {}), "sarsa": ("sarsa", {}), "expected_sarsa": ("expected_sarsa", {}), "q_learning": ("q_learning", {}),
            "double_q": ("double_q", {}), "nstep_sarsa_2": ("nstep_sarsa", {"n": 2}), "nstep_sarsa_3": ("nstep_sarsa", {"n": 3}),
            "dyna_q_n5": ("dyna_q", {"planning": 5}), "dyna_q_n20": ("dyna_q", {"planning": 20}),
            "prioritized_sweeping": ("prioritized_sweeping", {"planning": 10})}
best_method = next(m for m in by_method.index if m in LEARNERS)
best_seed = int(sel[sel.method == best_method].sort_values("val_reward").iloc[-1].seed)
final_policy = runs[(best_method, best_seed)]["policy"]
print(f"selected agent: {best_method} (seed {best_seed})")
save_json({"method": best_method, "seed": best_seed, "K": K, "policy": [int(c) for c in final_policy],
           "labels": [prob.space.label(int(c)) for c in final_policy]}, P["results"] / "policies" / "rl_full.json")
prob.describe_policy(final_policy)

In [ ]:
algo, hp = LEARNERS[best_method]
W0 = CFG["reward"]["weights"]
A = CFG["actions"]
fusion_noW = [A["fusion"].index(f) for f in A["fusion"] if "W" not in f]
variants = {
    "full RL (proposed)": {},
    "fixed wavelet (db6)": {"allowed": {"wavelet": "baseline"}},
    "fixed band (20-800 Hz)": {"allowed": {"band": "baseline"}},
    "fixed window (2 s)": {"allowed": {"window_s": "baseline"}},
    "no wavelet branch": {"allowed": {"fusion": fusion_noW}},
    "accuracy-only reward": {"weights": {"balanced": 1.0, "sensitivity": 0.0, "specificity": 0.0, "robustness": 0.0, "complexity": 0.0}},
    "no robustness term": {"weights": {**W0, "robustness": 0.0}},
    "no complexity penalty": {"weights": {**W0, "complexity": 0.0}},
    "no state adaptation (K=1)": {"K": 1},
    "classical reward oracle": {"oracle": "classical"},
}
abl_rows, abl_pol = [], {}
for vname, spec in variants.items():
    pr = RLProblem(CFG, P, K=spec.get("K", K), oracle=spec.get("oracle"))
    if "weights" in spec:
        pr.set_reward(spec["weights"])
    allowed = pr.allowed_fixed(**spec["allowed"]) if "allowed" in spec else None
    ref = RLProblem(CFG, P, K=spec.get("K", K))
    best = None
    for seed in R["seeds"]:
        out = run_tabular(pr.mdp("train", allowed=allowed, seed=seed), algo, R["episodes"], alpha=R["alpha"],
                          eps_sched=R["epsilon"], eval_every=0, **hp)
        pol = out["policy"]
        row = {"variant": vname, "seed": seed}
        for s in ("val", "test"):
            m = ref.evaluate(pol, s)
            row.update({f"{s}_{k}": m[k] for k in ("mean_reward", "macro_f1", "sensitivity", "specificity", "balanced_acc", "auroc")})
        mn = ref.evaluate(pol, "test", noisy=True)
        row.update({"noisy_test_macro_f1": mn["macro_f1"], "noisy_test_balanced_acc": mn["balanced_acc"]})
        abl_rows.append(row)
        if best is None or row["val_mean_reward"] > best[0]:
            best = (row["val_mean_reward"], pol, spec.get("K", K))
    abl_pol[vname] = best
for s in ("val", "test"):
    m = prob.evaluate(np.full(K, prob.space.baseline), s)
    mn = prob.evaluate(np.full(K, prob.space.baseline), "test", noisy=True)
    abl_rows.append({"variant": "no RL (fixed baseline config)", "seed": 0, **({f"{s}_{k}": m[k] for k in ("mean_reward", "macro_f1", "sensitivity", "specificity", "balanced_acc", "auroc")}),
                     "noisy_test_macro_f1": mn["macro_f1"], "noisy_test_balanced_acc": mn["balanced_acc"]})
abl = pd.DataFrame(abl_rows).groupby("variant", sort=False).mean(numeric_only=True).drop(columns="seed")
abl.to_csv(P["tables"] / "14_rl_ablations_proxy.csv")
abl.round(4)

In [ ]:
slug = {"full RL (proposed)": "rl_full", "fixed wavelet (db6)": "abl_fixed_wavelet", "fixed band (20-800 Hz)": "abl_fixed_band",
        "fixed window (2 s)": "abl_fixed_window", "no wavelet branch": "abl_no_wavelet_branch", "accuracy-only reward": "abl_accuracy_reward",
        "no robustness term": "abl_no_robustness", "no complexity penalty": "abl_no_complexity", "no state adaptation (K=1)": "abl_K1",
        "classical reward oracle": "abl_classical_oracle"}
for vname, (v, pol, KK) in abl_pol.items():
    if slug[vname] == "rl_full":
        continue
    save_json({"variant": vname, "K": KK, "policy": [int(c) for c in pol], "labels": [prob.space.label(int(c)) for c in pol]},
              P["results"] / "policies" / f"{slug[vname]}.json")
fig, ax = plt.subplots(figsize=(10, 4.5))
plot = abl[["val_macro_f1", "test_macro_f1", "noisy_test_macro_f1"]]
plot.plot.barh(ax=ax); ax.set_xlim(plot.min().min() - 0.05, plot.max().max() + 0.02); ax.set_title("RL ablations (proxy classifier, mean of 3 seeds)")
plt.tight_layout(); savefig(fig, "14_rl_ablations"); plt.show()